# MECG: Multilingual Evidence-Grounded Complaint Generation
## Notebook 01: Dataset Exploration & Forensic Audit

**Goal**: Explore the raw `Com-VID_data.csv` dataset, inspect column formats, audit missing values, investigate duplicate video URLs and IDs, analyze rating HTML tags, study emotion and aspect distributions, and identify critical data hygiene rules.

**Key Rules Examined**:
1. Inspect the actual CSV structure before implementing data transformations.
2. Preserve original columns and values.
3. Identify duplicate IDs and shared video URLs to prevent video-level data leakage.
4. Do not assume `Text` is an audio transcript or that `Ground-truth` is entirely visually supported.
5. Kaggle environment support (`/kaggle/input` and local fallbacks).

In [ ]:
# Environment Setup and Dynamic Path Detection (Kaggle vs Local)
import os
import sys
from pathlib import Path

# Auto-detect workspace root
CURRENT_DIR = Path.cwd()
KAGGLE_INPUT_DIR = Path('/kaggle/input/comvid-dataset')
LOCAL_DATA_DIR = Path('data')

if (KAGGLE_INPUT_DIR / 'Com-VID_data.csv').exists():
    RAW_CSV_PATH = KAGGLE_INPUT_DIR / 'Com-VID_data.csv'
elif (LOCAL_DATA_DIR / 'Com-VID_data.csv').exists():
    RAW_CSV_PATH = LOCAL_DATA_DIR / 'Com-VID_data.csv'
elif (Path('..') / 'data' / 'Com-VID_data.csv').exists():
    RAW_CSV_PATH = Path('..') / 'data' / 'Com-VID_data.csv'
else:
    RAW_CSV_PATH = Path('Com-VID_data.csv')

print(f'Using Raw Dataset Path: {RAW_CSV_PATH.resolve()}')
assert RAW_CSV_PATH.exists(), f'File not found: {RAW_CSV_PATH}'

### 1. Load Raw CSV and Inspect Column Headers
We inspect the raw field names, noticing any whitespace anomalies (such as `'Ground-truth '`).

In [ ]:
import csv
import json

with open(RAW_CSV_PATH, mode='r', encoding='utf-8', errors='replace') as f:
    reader = csv.DictReader(f)
    raw_fields = reader.fieldnames
    raw_rows = list(reader)

print(f'Total records in dataset: {len(raw_rows)}')
print('Raw CSV Field Names:')
for i, field in enumerate(raw_fields, 1):
    print(f'  {i}. {repr(field)} (length={len(field)})')

print('\nSample Record (Row 0):')
print(json.dumps(raw_rows[0], indent=2))

### 2. Forensic Audit of Identifiers & Video URLs
In video research, **video-level data leakage** occurs when multiple reviews point to the exact same video, and are inadvertently split across train and test sets. Let us audit ID and URL uniqueness.

In [ ]:
from collections import Counter

ids = [r['ID'].strip() for r in raw_rows if r.get('ID')]
id_counts = Counter(ids)
duplicate_ids = {k: v for k, v in id_counts.items() if v > 1}

urls = [r['Video-Source'].strip() for r in raw_rows if r.get('Video-Source')]
url_counts = Counter(urls)
duplicate_urls = {k: v for k, v in url_counts.items() if v > 1}
empty_urls = [r['ID'] for r in raw_rows if not r.get('Video-Source', '').strip()]

print(f'Total rows: {len(raw_rows)}')
print(f'Unique IDs: {len(id_counts)} | Duplicate IDs: {len(duplicate_ids)}')
print(f'Unique URLs: {len(url_counts)} | Duplicate URLs: {len(duplicate_urls)}')
print(f'Records missing Video-Source URL: {len(empty_urls)} -> IDs: {empty_urls}')

print('\nTop 3 Shared Video URLs across multiple reviews:')
for url, count in url_counts.most_common(3):
    matching_ids = [r['ID'] for r in raw_rows if r['Video-Source'].strip() == url]
    print(f'  - Count: {count} | Matching IDs: {matching_ids}')
    print(f'    URL: {url[:80]}...')

### 3. Rating Extraction from HTML
Amazon ratings are stored as HTML tags like `<span class="a-icon-alt">2.0 out of 5 stars</span>`. We extract numeric star ratings using regular expressions.

In [ ]:
import re

def extract_star_rating(html_str):
    if not html_str:
        return None
    match = re.search(r'([0-9]+(?:\.[0-9]+)?)\s*(?:out of|\/)\s*5', html_str, re.IGNORECASE)
    return float(match.group(1)) if match else None

raw_ratings = Counter(r['Rating'].strip() for r in raw_rows)
print('Raw HTML Rating strings:')
for rating_html, cnt in raw_ratings.items():
    parsed = extract_star_rating(rating_html)
    print(f'  Count: {cnt:4d} | Parsed: {parsed} stars | Raw: {repr(rating_html)}')

parsed_counts = Counter(extract_star_rating(r['Rating']) for r in raw_rows)
print('\nParsed Ratings Summary:', dict(parsed_counts))

### 4. Domain & Emotion Distribution
Let's explore the product domains and the emotion taxonomy. Note the capitalization inconsistency in emotion labels (e.g., `'blame'` vs `'Blame'`).

In [ ]:
domain_counts = Counter(r['Domain'].strip() for r in raw_rows)
print('Product Domain Distribution:')
for domain, cnt in domain_counts.most_common():
    print(f'  {domain:20s}: {cnt:4d} ({cnt/len(raw_rows)*100:.2f}%)')

emotion_counts = Counter(r['Emotion'].strip() for r in raw_rows)
print('\nEmotion Distribution (Raw):')
for emotion, cnt in emotion_counts.most_common():
    print(f'  {repr(emotion):20s}: {cnt:4d} ({cnt/len(raw_rows)*100:.2f}%)')

# Normalized emotion
norm_emotion = Counter(r['Emotion'].strip().lower() for r in raw_rows)
print('\nEmotion Distribution (Normalized to lowercase):')
for emotion, cnt in norm_emotion.most_common():
    print(f'  {emotion:20s}: {cnt:4d} ({cnt/len(raw_rows)*100:.2f}%)')

### 5. Complaint Aspect Analysis & Typo Identification
Aspect labels contain multiple comma/semicolon-separated labels, plus spelling variations and typos. Rule: Fix typos without merging semantically distinct aspects.

In [ ]:
all_aspect_tokens = Counter()
empty_aspect_rows = []

for idx, r in enumerate(raw_rows):
    aspect_str = r.get('Aspect', '').strip()
    if not aspect_str:
        empty_aspect_rows.append(r['ID'])
        continue
    tokens = re.split(r'[,;/]|\band\b', aspect_str, flags=re.IGNORECASE)
    for tok in tokens:
        t = tok.strip().rstrip('.')
        if t:
            all_aspect_tokens[t] += 1

print(f'Total distinct raw aspect tokens: {len(all_aspect_tokens)}')
print(f'Records with empty Aspect field: {len(empty_aspect_rows)} (IDs: {empty_aspect_rows[:5]}...)')

print('\nTop 15 Aspect Tokens:')
for tok, count in all_aspect_tokens.most_common(15):
    print(f'  {tok:25s}: {count:4d}')

print('\nNotice Typo Examples that must be normalized:')
typo_examples = ['Qualtiy', 'Qualiy', 'Usabilty', 'Usabiity', 'Manufaturing', 'Functionaliy']
for typo in typo_examples:
    if typo in all_aspect_tokens:
        print(f'  Typo: {typo!r} (occurs {all_aspect_tokens[typo]} times)')

### 6. Customer Text vs Reference Ground-Truth Complaint
Examining the customer text against the reference ground truth.
Important research takeaway: Customer text is often very terse (e.g. 'Not recommended'), while `Ground-truth` contains specific details observed from video or inspection.

In [ ]:
text_lens = [len(r['Text'].split()) for r in raw_rows]
gt_col = 'Ground-truth ' if 'Ground-truth ' in raw_rows[0] else 'Ground-truth'
gt_lens = [len(r[gt_col].split()) for r in raw_rows]

print('Customer Text Word Length:')
print(f'  Min: {min(text_lens)}, Max: {max(text_lens)}, Mean: {sum(text_lens)/len(text_lens):.1f}')

print('Ground-truth Complaint Word Length:')
print(f'  Min: {min(gt_lens)}, Max: {max(gt_lens)}, Mean: {sum(gt_lens)/len(gt_lens):.1f}')

print('\nExample 1 where Customer Text is short but Ground Truth is descriptive:')
for r in raw_rows:
    if len(r['Text'].split()) <= 3:
        print(f"ID: {r['ID']}")
        print(f"Customer Text: {repr(r['Text'])}")
        print(f"Ground-truth : {repr(r[gt_col])}")
        break

### Summary of Exploratory Findings
- Total Records: 1,175 rows.
- 11 Duplicate Review IDs and 86 Shared Video URLs require **Group-Aware Splitting** to eliminate data leakage.
- 3 rows lack a video URL.
- 16 rows lack aspect labels.
- Emotion has 4 underlying classes (`blame`, `disappointment`, `dissatisfaction`, `frustration`) with minor casing anomalies.
- Ratings are predominantly 1.0 star (82.5%) and 2.0 stars (17.4%).
- All raw values must be preserved under `raw_*` columns during cleaning.

Proceed to **Notebook 02: Dataset Cleaning** to generate the cleaned dataset and audit report.